# Matrix Factorisation on Movie Lens 1M dataset
Dataset from: [Movie Lens 1M Dataset](https://grouplens.org/datasets/movielens/1m/)

In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np
import re
import seaborn as sns
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from wordcloud import WordCloud

### Load dataset

In [ ]:
def load_movielens_movies(path: str = "Dataset/ml-1m/movies.dat") -> pd.DataFrame:
    return pd.read_csv(
        path,
        sep="::",
        engine="python",
        names=["movie_id", "title", "genres"],
        encoding="latin-1",
    )

def load_movielens_ratings(path: str = "Dataset/ml-1m/ratings.dat") -> pd.DataFrame:
    return pd.read_csv(
        path,
        sep="::",
        engine="python",
        names=["user_id", "movie_id", "rating", "timestamp"],
        encoding="latin-1",
    )

def preprocess_data(split_titles=False):
    """
    Merges movies and ratings on movieId.
    """
    movies = load_movielens_movies()
    # Split title and year out of title
    if split_titles:
        movies["year"] = movies["title"].apply(lambda movie_name: re.search('\\((\\d*)\\)', movie_name).groups(1)[0])
        movies["title"] = movies["title"].apply(lambda movie_name: movie_name.split(" (")[0])
        # TODO - fix what happens for duplicated movie titles
    # Get ratings
    ratings = load_movielens_ratings()
    merged = pd.merge(ratings, movies, on="movie_id")
    # Convert timestamps from unix
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], unit='s')
    return merged

In [ ]:
df = preprocess_data()

In [ ]:
df.head()

### EDA

Ref:
- [MovieLens-1M Deep Dive – Part I](https://towardsdatascience.com/movielens-1m-deep-dive-part-i-8acfeda1ad4/)

In [ ]:
df.rating.value_counts().sort_index().plot(kind="bar", xlabel="Rating", ylabel="Count")

In [ ]:
df.timestamp.groupby(df.timestamp.dt.year).count().plot(kind="bar", xlabel="Rating Timestamp", ylabel="Count")

In [ ]:
movies_df = df.copy(deep=True)

In [ ]:
movies_df['genres'] = movies_df['genres'].apply(lambda x: x.split('|'))
movies_df_exploded = movies_df.explode("genres")
px.histogram(movies_df_exploded, x="genres", height=400, title="Movie count by genre").update_xaxes(categoryorder="total descending")

In [ ]:
rating_by_genre_df = movies_df_exploded.groupby('genres').agg({'rating': ['mean', 'count']}).sort_values(('rating', 'mean')).reset_index()
rating_by_genre_df.columns = ['_'.join(col).strip() for col in rating_by_genre_df.columns.values]
px.bar(rating_by_genre_df, x='genres_', y='rating_mean', height=300)

In [ ]:
movies_df["year"] = movies_df["title"].apply(lambda movie_name: re.search('\\((\\d*)\\)', movie_name).groups(1)[0])
movie_count_by_year = px.histogram(movies_df, x='year', height=400, title='Movie count by year').update_xaxes(categoryorder="category ascending")
movie_count_by_year

## Matrix Factorisation

### Similarity

#### Pivot table + SKLearn NMF
Using:
- [SKLearn NMF (Non-Negative Matrix Factorisation)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)]
- Similarity code from [here](https://github.com/dinesh-git17/movie_recommendation/tree/main)
- Top-N code from [here](https://medium.com/@quindaly/step-by-step-nmf-example-in-python-9974e38dc9f9)

In [ ]:
class MatrixFactorisation:
    def __init__(self, data, min_ratings=100, n_components=20):
        self._create_pivot_table(data, min_ratings)
        self._NMF_model()

    def _create_pivot_table(self, data, min_ratings=100):
        """
        Creates a pivot table (users x movies) with ratings.
        Only movies with at least min_ratings are retained.

        Arguments:
            data (): TBC
            min_ratings (int): TBC
        Returns:
            TBC
        """
        # Count number of ratings per movie
        ratings_count = data.groupby("title")["rating"].count()
        popular_movies = ratings_count[ratings_count >= min_ratings].index
        filtered_data = data[data["title"].isin(popular_movies)]

        # Create pivot table: rows = user_id, columns = title, values = rating
        pivot = filtered_data.pivot_table(index="user_id", columns="title", values="rating")
        # Fill missing values with 0
        pivot_filled = pivot.fillna(0)
        self.pivot = pivot_filled


    def _rank_calculation(self):
        """
        Calculate the optimal rank of the specified dataframe.
        """
        # Calculate benchmark value
        benchmark = np.linalg.norm(self.pivot, ord='fro') * 0.0001

        # Iterate through various values of rank to find optimal
        rank = 3
        while True:
            # initialize the model
            model = NMF(n_components=rank, init='random', random_state=0, max_iter=1000)
            W = model.fit_transform(self.pivot)
            H = model.components_
            V = W @ H

            # Calculate RMSE of original table and new V
            RMSE = np.sqrt(mean_squared_error(self.pivot, V))

            if RMSE < benchmark:
                return rank, V

            # Increment rank if RMSE isn't smaller than the benchmark
            rank += 1


    def _NMF_model(self):
        """
        Generates recommendations using NMF-based matrix factorization
        Fills missing ratings with 0, factorizes the matrix, and computes cosine similarities
        on the movie latent factors.

        Arguments:
            TBC
        Returns:
            recommendations (): top_n recommendations based on similarity to movie_id
        """
        # Apply NMF to factorize the matrix into user and movie latent factors
        optimal_rank = self._rank_calculation()
        print(optimal_rank)
        nmf_model = NMF(n_components=optimal_rank, init="random", random_state=42, max_iter=1000)
        self.W = nmf_model.fit_transform(self.pivot)
        self.H = nmf_model.components_  # shape: (n_components, n_movies)


    def user_top_N(self, user_id, top_n=10):
        """TBC"""
        W_df = pd.DataFrame(self.W)
        H_df = pd.DataFrame(self.H)
        V = pd.DataFrame(np.dot(W_df,H_df), columns=self.pivot.columns)
        V.index = self.pivot.index

        if user_id not in V.index:
            raise ValueError(f"User with ID '{user_id}' not found in the dataset.")

        # Top N movies user hasn't reviewed
        V_T = V.T
        V_T.to_csv("V.csv")
        pivot_T = self.pivot.T
        pivot_T.to_csv("pivot.csv")
        user_ratings = V_T[user_id].sort_values(ascending=False)
        user_ranking = [movie for movie in user_ratings.index if pivot_T[user_id].loc[movie] == 0]
        return user_ranking[:top_n]


    def understand_user_profile(self, user_id, movies_df):
        """TBC"""
        # Extract movies rated by user
        pivot_T = self.pivot.T
        true_user_ratings = pivot_T[user_id].sort_values(ascending=False)
        true_user_ranking = [movie for movie in true_user_ratings.index if pivot_T[user_id].loc[movie] != 0]

        # Get genres of movies rated
        genres = []
        for movie in true_user_ranking:
            index = movies_df.title[movies_df.title == movie].index.to_list()[0]
            [genres.append(genre) for genre in movies_df["genres"][index]]

        # TODO - other ideas: average rating for genres plot, distribution of scores etc

        # Make wordcloud of genres
        genres_string=(" ").join(genres)
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate(genres_string)
        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.show()


    def movie_similarity(self, movie_title, num_rec=10):
        """TBC"""
        # Transpose H to get movie latent factors: shape (n_movies, n_components)
        movie_factors = self.H.T
        titles = self.pivot.columns.tolist()

        # Compute cosine similarity between movies using the latent factors
        similarity_matrix = cosine_similarity(movie_factors)
        similarity_df = pd.DataFrame(
            similarity_matrix, index=titles, columns=titles
        )

        if movie_title not in similarity_df.index:
            raise ValueError(f"Movie '{movie_title}' not found in the dataset.")

        # Get the similarity series for the given movie and sort descending
        similar_movies = (
            similarity_df[movie_title]
            .drop(labels=[movie_title])
            .sort_values(ascending=False)
        )
        recommendations = similar_movies.head(num_rec)
        return recommendations

In [ ]:
NMF_model = MatrixFactorisation(df)

In [ ]:
recs = NMF_model.movie_similarity("101 Dalmatians (1961)")
recs

In [ ]:
recs = NMF_model.movie_similarity("101 Dalmatians (1996)")
recs

In [ ]:
recs = NMF_model.movie_similarity("10 Things I Hate About You (1999)")
recs

In [ ]:
recs = NMF_model.movie_similarity("Young Guns (1988)")
recs

Thoughts:
* Not recommending sequels
* Not using a test-train split -> this algorithm won't work if the requested movie doesn't exist in the pivot table
* Therefore, can't handle new movies or users

In [ ]:
# TODO - metric based evaluation

In [ ]:
user_id  = 44
NMF_model.understand_user_profile(user_id, movies_df)
rec = NMF_model.user_top_N(user_id)
rec

Splitting out years:
- Can't as there are some films with the same titles which are unique only ecause of year (e.g. 101 Dalmatians)

BUT! Algorithm doesn't use movie titles just ratings to get factorised components.
So shouldn't matter.